# Repository guide: Fadhma full training

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution. This recovered notebook has no saved cell outputs. Separate training history and experiment summary are included.


# Fadhma-300M → Tarifit V1.2 Transfer Fine-Tuning

This notebook evaluates **related-language transfer from Kabyle to Tarifit**.

**Starting checkpoint:** `agbalu/Fadhma-300M`  
**Original SSL backbone:** `ylacombe/omniASR_W2V_300M_SSL`  
**Target:** Tarifit V1.2

This experiment is deliberately matched to the OmniASR V1.2 run: same frozen 1,754/129 split, same 34-token Tarifit vocabulary, no augmentation, same optimizer schedule, same maximum 8 epochs, same early stopping, same greedy decoding, and checkpoint selection by validation CER.

The original Kabyle CTC head is replaced because its label inventory differs from Tarifit V1.2. The Kabyle-adapted acoustic encoder is retained and fine-tuned toward Tarifit with a fresh 34-class CTC head.


In [ ]:
# Cell 1 — Install the exact experiment dependencies

!pip -q install \
    "transformers==4.57.1" \
    "datasets==4.4.1" \
    "accelerate>=1.10,<2" \
    "jiwer==4.0.0" \
    "safetensors>=0.4.5" \
    "soundfile>=0.12.1"

print("✓ Dependencies installed.")
print("If Colab requests a restart after installation, restart once and continue from Cell 2.")


In [ ]:
# Cell 2 — Mount Drive and define Fadhma transfer experiment paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = (
    PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2_train_val_frozen.csv"
)

TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

OUTPUT_DIR = PROJECT_ROOT / "models" / "fadhma_300m_tarifit_v1_2_transfer"
RESULTS_DIR = PROJECT_ROOT / "results" / "fadhma_300m_tarifit_v1_2_transfer"

OMNI_SUMMARY_PATH = (
    PROJECT_ROOT / "results" / "omniASR_w2v_300m_tarifit_v1_2" / "experiment_summary.json"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = "agbalu/Fadhma-300M"
SOURCE_BASE_MODEL = "ylacombe/omniASR_W2V_300M_SSL"

print("Project:", PROJECT_ROOT)
print("Transfer checkpoint:", BASE_MODEL_ID)
print("Original SSL backbone:", SOURCE_BASE_MODEL)
print("Output:", OUTPUT_DIR)
print("Results:", RESULTS_DIR)


In [ ]:
# Cell 3 — Verify software versions and GPU

import sys
import json
import random
import hashlib
import platform
import numpy as np
import pandas as pd
import torch
import transformers
import datasets

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

assert transformers.__version__ == "4.57.1"
assert datasets.__version__ == "4.4.1"

if not torch.cuda.is_available():
    raise RuntimeError("Please switch Colab to a GPU runtime before continuing.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)


In [ ]:
# Cell 4 — Load the exact frozen V1.2 train/validation split

assert FROZEN_METADATA_PATH.exists(), (
    f"Frozen metadata not found: {FROZEN_METADATA_PATH}"
)

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

for col in [
    "transcription",
    "final_selection",
    "review_status",
    "dataset_split",
]:
    if col in frozen_df.columns:
        frozen_df[col] = (
            frozen_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

frozen_df["dataset_split"] = frozen_df["dataset_split"].str.lower()

train_selected = frozen_df[
    frozen_df["dataset_split"].eq("train")
].copy()

val_selected = frozen_df[
    frozen_df["dataset_split"].eq("validation")
].copy()

selected = pd.concat(
    [train_selected, val_selected],
    ignore_index=True,
)

assert len(train_selected) == 1754, (
    f"Expected 1754 train segments, found {len(train_selected)}"
)
assert len(val_selected) == 129, (
    f"Expected 129 validation segments, found {len(val_selected)}"
)
assert len(selected) == 1883
assert not selected["segment_id"].duplicated().any()

print("Train segments:", len(train_selected))
print("Validation segments:", len(val_selected))
print(
    "Train hours:",
    round(train_selected["duration_seconds"].sum() / 3600, 3),
)
print(
    "Validation hours:",
    round(val_selected["duration_seconds"].sum() / 3600, 3),
)
print("Train speakers:", train_selected["speaker_group_id"].nunique())
print("Validation speakers:", val_selected["speaker_group_id"].nunique())

print("\n✓ Exact frozen V1.2 split loaded.")


In [ ]:
# Cell 5 — Verify speaker independence and test isolation

train_speakers = set(train_selected["speaker_group_id"].dropna())
val_speakers = set(val_selected["speaker_group_id"].dropna())

print("Train/validation speaker overlap:", bool(train_speakers & val_speakers))

assert not (train_speakers & val_speakers), (
    f"Speaker leakage detected: {train_speakers & val_speakers}"
)

# Inspect the master metadata only for held-out test speakers.
master_df = pd.read_csv(METADATA_PATH)

for col in ["dataset_split", "speaker_group_id"]:
    master_df[col] = (
        master_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

test_speakers = set(
    master_df.loc[
        master_df["dataset_split"].str.lower().eq("test"),
        "speaker_group_id",
    ]
)

print("Train/test speaker overlap:", bool(train_speakers & test_speakers))
print("Validation/test speaker overlap:", bool(val_speakers & test_speakers))

assert not (train_speakers & test_speakers)
assert not (val_speakers & test_speakers)

print("✓ No speaker leakage.")


In [ ]:
# Cell 6 — Verify the final V1.2 character inventory

import unicodedata
from collections import Counter

FINAL_LETTERS = [
    "a", "b", "c", "d", "ḍ", "e", "ɛ", "f", "g", "h", "ḥ",
    "i", "j", "k", "l", "m", "n", "p", "q", "r", "s", "t",
    "ṭ", "u", "v", "w", "x", "y", "z", "ɣ", "ʷ",
]

allowed_chars = set(FINAL_LETTERS) | {" "}

counter = Counter(
    ch
    for text in selected["transcription"].astype(str)
    for ch in text
)

unexpected = sorted(
    ch
    for ch in counter
    if ch not in allowed_chars
)

combining_marks = {
    ch: count
    for ch, count in counter.items()
    if unicodedata.combining(ch)
}

print("Final letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected)
print("Combining marks:", combining_marks)

assert not unexpected
assert not combining_marks

print("✓ Final V1.2 orthography verified.")


In [ ]:
# Cell 7 — Verify frozen metadata hash

sha256 = hashlib.sha256(
    FROZEN_METADATA_PATH.read_bytes()
).hexdigest()

print("Frozen rows:", len(frozen_df))
print("SHA256:", sha256)
print("✓ Frozen metadata fingerprint recorded.")


In [ ]:
# Cell 8 — Load the exact existing V1.2 tokenizer

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
assert VOCAB_PATH.exists(), f"Missing tokenizer: {VOCAB_PATH}"

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    existing_vocab = json.load(f)

expected_tokens = set(FINAL_LETTERS) | {"|", "[UNK]", "[PAD]"}
saved_tokens = set(existing_vocab.keys())

missing_tokens = sorted(expected_tokens - saved_tokens)
unexpected_tokens = sorted(saved_tokens - expected_tokens)

print("Vocabulary size:", len(existing_vocab))
print("Missing tokens:", missing_tokens)
print("Unexpected tokens:", unexpected_tokens)

assert len(existing_vocab) == 34, f"Expected 34 tokens, found {len(existing_vocab)}"
assert not missing_tokens, f"Missing expected tokens: {missing_tokens}"
assert not unexpected_tokens, f"Unexpected tokenizer tokens: {unexpected_tokens}"
assert sorted(existing_vocab.values()) == list(range(34)), "Tokenizer IDs are not 0..33."

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Tokenizer size:", len(tokenizer))
print("PAD / CTC blank id:", tokenizer.pad_token_id)
print("UNK id:", tokenizer.unk_token_id)
print("Word delimiter id:", tokenizer.convert_tokens_to_ids("|"))

assert len(tokenizer) == 34
print("✓ Exact existing V1.2 tokenizer reused.")


In [ ]:
# Cell 9 — Verify zero unknown tokens in the frozen references

unk_id = tokenizer.unk_token_id
unknown_segments = []

for row in frozen_df.itertuples(index=False):
    ids = tokenizer(row.transcription).input_ids
    if unk_id in ids:
        unknown_segments.append(
            (row.segment_id, row.transcription)
        )

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    display(pd.DataFrame(
        unknown_segments[:30],
        columns=["segment_id", "transcription"],
    ))

assert not unknown_segments

print("✓ All references are representable by the tokenizer.")


In [ ]:
# Cell 10 — Reuse the exact V1.2 cached waveforms and labels

from datasets import load_from_disk

assert DATASET_CACHE_DIR.exists(), (
    f"Missing cached dataset: {DATASET_CACHE_DIR}"
)
assert CACHE_MANIFEST_PATH.exists(), (
    f"Missing cache manifest: {CACHE_MANIFEST_PATH}"
)

with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as f:
    cache_manifest = json.load(f)

assert cache_manifest.get("metadata_sha256") == sha256, (
    "Cached dataset does not match the frozen metadata."
)

dataset = load_from_disk(str(DATASET_CACHE_DIR))
train_ds = dataset["train"]
val_ds = dataset["validation"]

assert len(train_ds) == 1754
assert len(val_ds) == 129

print("Train examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Columns:", train_ds.column_names)
print("✓ Exact V1.2 cached data reused.")


In [ ]:
# Cell 11 — Show duration statistics

def duration_summary(frame):
    seconds = frame["duration_seconds"].astype(float)

    return {
        "segments": len(frame),
        "hours": seconds.sum() / 3600,
        "min_seconds": seconds.min(),
        "mean_seconds": seconds.mean(),
        "max_seconds": seconds.max(),
    }

train_duration = duration_summary(train_selected)
val_duration = duration_summary(val_selected)

display(pd.DataFrame([
    {"split": "train", **train_duration},
    {"split": "validation", **val_duration},
]))


In [ ]:
# Cell 12 — Load Fadhma-300M and initialize a fresh Tarifit CTC head

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,

    # Fadhma has a Kabyle CTC inventory. Retain its Kabyle-adapted encoder
    # but initialize a new target head for the 34-token Tarifit V1.2 vocabulary.
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,

    # Match the OmniASR V1.2 comparison: no augmentation.
    apply_spec_augment=False,
)

# Preserve Fadhma's transfer setup: freeze the convolutional feature extractor,
# while fine-tuning the Transformer encoder and fresh Tarifit CTC head.
model.freeze_feature_encoder()
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print("Target CTC vocabulary size:", model.config.vocab_size)
print("SpecAugment enabled:", model.config.apply_spec_augment)

assert model.config.vocab_size == 34
assert model.config.apply_spec_augment is False
assert 300_000_000 < total_params < 330_000_000
assert 300_000_000 < trainable_params < total_params

print("\n✓ Kabyle-adapted Fadhma encoder loaded.")
print("✓ Fresh 34-class Tarifit CTC head initialized.")
print("✓ Convolutional feature extractor frozen.")


In [ ]:
# Cell 13 — Run exact CTC feasibility checks for Fadhma

def minimum_ctc_frames(labels):
    repeats = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats


def model_output_frames(input_samples):
    return int(
        model._get_feat_extract_output_lengths(
            torch.tensor(int(input_samples))
        ).item()
    )


def find_ctc_infeasible(split_ds):
    bad = []

    for i, example in enumerate(split_ds):
        output_frames = model_output_frames(
            example["input_length"]
        )

        min_frames = minimum_ctc_frames(
            example["labels"]
        )

        if output_frames < min_frames:
            bad.append({
                "index": i,
                "segment_id": example["segment_id"],
                "input_samples": example["input_length"],
                "audio_seconds": example["input_length"] / 16000,
                "output_frames": output_frames,
                "label_length": len(example["labels"]),
                "minimum_ctc_frames": min_frames,
            })

    return bad


bad_train = find_ctc_infeasible(train_ds)
bad_val = find_ctc_infeasible(val_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_val))

if bad_train:
    display(pd.DataFrame(bad_train))
if bad_val:
    display(pd.DataFrame(bad_val))

assert not bad_train, (
    "Training contains CTC-infeasible examples."
)
assert not bad_val, (
    "Validation contains CTC-infeasible examples."
)

print("✓ All 1754/129 examples are CTC-feasible for Fadhma.")


In [ ]:
# Cell 14 — Define the dynamic CTC padding collator

from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        batch["labels"] = labels_batch[
            "input_ids"
        ].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor
)

print("✓ CTC collator ready.")


In [ ]:
# Cell 15 — Define WER and CER

from jiwer import wer, cer

def compute_metrics(pred):
    pred_ids = np.argmax(
        pred.predictions,
        axis=-1,
    )

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = [
        x.strip()
        for x in processor.batch_decode(pred_ids)
    ]

    ref_str = [
        x.strip()
        for x in processor.batch_decode(
            label_ids,
            group_tokens=False,
        )
    ]

    return {
        "wer": wer(ref_str, pred_str),
        "cer": cer(ref_str, pred_str),
    }

print("✓ WER/CER metric function ready.")


In [ ]:
# Cell 16 — Set reproducibility seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


In [ ]:
# Cell 17 — Configure the matched Fadhma transfer training schedule

from transformers import (
    TrainingArguments,
    EarlyStoppingCallback,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Match OmniASR V1.2 so initialization is the main experimental difference.
    num_train_epochs=8,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    save_total_limit=2,

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    dataloader_num_workers=2,
    remove_unused_columns=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001,
)

print("Max epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Weight decay:", training_args.weight_decay)
print("Effective batch size:", 2 * 8)
print("Selection metric:", training_args.metric_for_best_model)
print("Early-stopping patience:", 2)
print("✓ Matched Fadhma/Omni training configuration ready.")


In [ ]:
# Cell 18 — Create the Fadhma transfer Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[early_stopping],
)

print("Trainer train examples:", len(trainer.train_dataset))
print("Trainer validation examples:", len(trainer.eval_dataset))

assert len(trainer.train_dataset) == 1754
assert len(trainer.eval_dataset) == 129

print("✓ Fadhma transfer Trainer ready.")


In [ ]:
# Cell 19 — Start or resume Fadhma → Tarifit V1.2 fine-tuning

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None
if OUTPUT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint is not None:
    print("Found checkpoint:", last_checkpoint)
    print("Resuming Fadhma transfer training from the latest checkpoint.")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No previous checkpoint found.")
    print("Starting fresh Fadhma → Tarifit V1.2 transfer fine-tuning.")
    train_result = trainer.train()

print("\nTraining finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


In [ ]:
# Cell 20 — Show and save epoch-by-epoch Fadhma results

rows = []
last_train_loss = None

for log in trainer.state.log_history:
    if (
        "loss" in log
        and "eval_loss" not in log
    ):
        last_train_loss = log["loss"]

    if "eval_loss" in log:
        rows.append({
            "epoch": log.get("epoch"),
            "training_loss": last_train_loss,
            "validation_loss": log.get("eval_loss"),
            "WER": log.get("eval_wer"),
            "CER": log.get("eval_cer"),
        })

history_df = pd.DataFrame(rows)

display(history_df)

HISTORY_PATH = RESULTS_DIR / "training_history.csv"

history_df.to_csv(
    HISTORY_PATH,
    index=False,
)

print("Saved:", HISTORY_PATH)


In [ ]:
# Cell 21 — Evaluate the selected best Fadhma checkpoint

best_metrics = trainer.evaluate(
    eval_dataset=val_ds
)

best_wer = best_metrics["eval_wer"]
best_cer = best_metrics["eval_cer"]

print(
    f"Best validation WER: {best_wer:.6f} "
    f"({best_wer * 100:.2f}%)"
)
print(
    f"Best validation CER: {best_cer:.6f} "
    f"({best_cer * 100:.2f}%)"
)
print(
    "Selected checkpoint:",
    trainer.state.best_model_checkpoint,
)


In [ ]:
# Cell 22 — Save the selected Fadhma → Tarifit model and processor

BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

trainer.save_model(
    str(BEST_MODEL_DIR)
)

processor.save_pretrained(
    str(BEST_MODEL_DIR)
)

print("Saved selected model:", BEST_MODEL_DIR)


In [ ]:
# Cell 23 — Save Fadhma validation predictions for qualitative error analysis

prediction_output = trainer.predict(
    val_ds
)

pred_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

label_ids = prediction_output.label_ids.copy()
label_ids[
    label_ids == -100
] = tokenizer.pad_token_id

predictions = [
    x.strip()
    for x in processor.batch_decode(pred_ids)
]

references = [
    x.strip()
    for x in processor.batch_decode(
        label_ids,
        group_tokens=False,
    )
]

predictions_df = pd.DataFrame({
    "segment_id": val_ds["segment_id"],
    "reference": references,
    "prediction": predictions,
})

PREDICTIONS_PATH = (
    RESULTS_DIR
    / "validation_predictions.csv"
)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(predictions_df.head(20))

print("Saved:", PREDICTIONS_PATH)


In [ ]:
# Cell 24 — Save the complete Fadhma transfer experiment summary

summary = {
    "experiment": "Fadhma-300M Kabyle -> Tarifit V1.2 transfer fine-tuning",
    "transfer_checkpoint": BASE_MODEL_ID,
    "source_language": "Kabyle",
    "target_language": "Tarifit",
    "original_ssl_backbone": SOURCE_BASE_MODEL,
    "data_version": "V1.2",
    "metadata_sha256": sha256,
    "train_segments": len(train_ds),
    "validation_segments": len(val_ds),
    "train_hours": train_duration["hours"],
    "validation_hours": val_duration["hours"],
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "augmentation": "none",
    "source_ctc_head_replaced": True,
    "feature_extractor_frozen": True,
    "encoder_finetuned": True,
    "seed": SEED,
    "max_epochs": 8,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "physical_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "early_stopping_patience": 2,
    "checkpoint_selection_metric": "CER",
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_wer": float(best_wer),
    "best_validation_cer": float(best_cer),
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved:", SUMMARY_PATH)


In [ ]:
# Cell 25 — Compare Fadhma transfer directly with OmniASR V1.2

comparison_rows = [
    {
        "experiment": "Fadhma-300M → Tarifit V1.2",
        "initialization": "Kabyle-adapted OmniASR backbone",
        "WER": float(best_wer),
        "CER": float(best_cer),
    }
]

if OMNI_SUMMARY_PATH.exists():
    with open(OMNI_SUMMARY_PATH, "r", encoding="utf-8") as f:
        omni_summary = json.load(f)

    comparison_rows.insert(
        0,
        {
            "experiment": "OmniASR-W2V-300M → Tarifit V1.2",
            "initialization": "multilingual SSL backbone",
            "WER": float(omni_summary["best_validation_wer"]),
            "CER": float(omni_summary["best_validation_cer"]),
        },
    )
else:
    print("Omni summary not found:", OMNI_SUMMARY_PATH)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

if len(comparison_df) == 2:
    omni_cer = comparison_df.iloc[0]["CER"]
    fadhma_cer = comparison_df.iloc[1]["CER"]
    delta = fadhma_cer - omni_cer

    print(
        "\nCER difference (Fadhma - Omni):",
        f"{delta:+.4f}",
        f"({delta * 100:+.2f} percentage points)",
    )

    if delta < 0:
        print("✓ Positive related-language transfer on validation: Fadhma beats Omni by CER.")
    elif delta > 0:
        print("Kabyle initialization did not beat direct Omni initialization by CER.")
    else:
        print("The two initializations tie by CER.")


## Interpretation

This is a controlled **initialization comparison** rather than simply another model leaderboard entry.

Fadhma-300M was itself fine-tuned from the same `ylacombe/omniASR_W2V_300M_SSL` backbone used in the direct OmniASR V1.2 experiment. The target corpus, target tokenizer, decoding method, optimizer schedule, validation set, and checkpoint-selection criterion are therefore matched.

- If Fadhma performs better, this supports **positive related-language transfer from Kabyle to Tarifit** under this setup.
- If it performs similarly, Kabyle specialization provides little measurable advantage.
- If it performs worse, Kabyle source-language specialization does not provide positive transfer under the chosen adaptation procedure.

Do not repeatedly evaluate on the held-out test set while selecting models. Final test evaluation should occur only after test references are manually finalized and frozen.
